# 📊 **Evaluación Sumativa Unidad 02: Inferencia Estadística y Diagnóstico Paramétrico (ABP)**

---
**Estudiante:** Pilar Valentina Naranjo Quizhpe  
**Carrera:** Computación – Segundo Ciclo  
**Período Académico:** 🗓️ Marzo 2026 – Agosto 2026  
**Docente:** Ing.  Cristian Narváez Guillén                                 


# 🧪 **Sección A: Prueba de Hipótesis Unimuestral**

En esta sección se analiza una variable representativa del Proyecto Integrador Regional mediante una **Prueba T de Student para una muestra**, con el objetivo de determinar si el promedio poblacional difiere significativamente de un valor de referencia.

---

**Contextualización del problema**

La disponibilidad de servicios básicos constituye uno de los principales indicadores del desarrollo de un territorio. En este análisis se estudia la variable **Falta de Alcantarillado** en los cantones de la provincia de Loja.

Para ello, se calcula el porcentaje de viviendas que no disponen de este servicio mediante el siguiente indicador:

$$
\text{Porcentaje de falta de alcantarillado} = \left( \frac{\text{Viviendas sin alcantarillado}}{\text{Total de viviendas}} \right) \times 100
$$

El propósito es determinar, mediante inferencia estadística, si el porcentaje promedio de viviendas sin alcantarillado difiere significativamente de una tasa de referencia del **30%**.

---

In [ ]:
import pandas as pd
from scipy import stats

# 1. Cargar el conjunto de datos
df_loja = pd.read_csv('/content/datos_loja2.csv')

# 2. Calcular el porcentaje de viviendas sin alcantarillado
df_loja['Pct_Sin_Alcantarillado'] = (
    df_loja['Sin_Alcantarillado'] / df_loja['Viviendas']
) * 100

# 3. Definir la hipótesis
mu_0 = 30.0
valores_alcantarillado = df_loja['Pct_Sin_Alcantarillado']

# 4. Ejecutar la prueba t de Student para una muestra
t_stat, p_value = stats.ttest_1samp(
    valores_alcantarillado,
    popmean=mu_0
)

# 5. Calcular estadísticas descriptivas
media_muestral = valores_alcantarillado.mean()
tamano_muestra = len(valores_alcantarillado)
grados_libertad = tamano_muestra - 1

# 6. Mostrar resultados
print("=" * 50)
print("PRUEBA DE HIPÓTESIS UNIMUESTRAL")
print("=" * 50)

print(f"Media muestral           : {media_muestral:.2f}%")
print(f"Valor de referencia (μ₀) : {mu_0:.2f}%")
print(f"Tamaño de la muestra (n) : {tamano_muestra}")
print(f"Grados de libertad (df)  : {grados_libertad}")
print(f"Estadístico t            : {t_stat:.4f}")
print(f"Valor-p                  : {p_value:.6f}")



PRUEBA DE HIPÓTESIS UNIMUESTRAL
Media muestral           : 34.65%
Valor de referencia (μ₀) : 30.00%
Tamaño de la muestra (n) : 16
Grados de libertad (df)  : 15
Estadístico t            : 2.7911
Valor-p                  : 0.013705


# 📊 **Justificación Estadística Basada en el Valor-p**

Se establece un nivel de significancia de $\alpha = 0.05$. El análisis realizado mediante la prueba t de Student para una muestra arrojó un estadístico $t = 2.7911$ y un valor-p $= 0.013705$.

Dado que el valor-p obtenido ($0.013705$) es menor que el nivel de significancia ($0.05$), se rechaza la hipótesis nula ($H_0$). Esto indica que existe evidencia estadísticamente suficiente para concluir que el porcentaje promedio de viviendas sin alcantarillado en los cantones de la provincia de Loja es diferente del valor de referencia del $30\%$.

Además, la media muestral calculada fue de $34.65\%$, superior al valor de referencia planteado. Esto sugiere que, en los cantones analizados, el porcentaje promedio de viviendas sin alcantarillado es mayor al esperado; sin embargo, la decisión estadística se fundamenta en que la diferencia observada es estadísticamente significativa, de acuerdo con el valor-p obtenido.

#📊 **Sección B: Comparación de Grupos mediante ANOVA de un Factor**

Con el objetivo de analizar si el porcentaje de viviendas sin alcantarillado presenta diferencias entre cantones con distintos niveles de población, los cantones de la provincia de Loja fueron clasificados en tres grupos utilizando los terciles de la variable población. Esta clasificación permitió formar grupos de tamaño similar denominados:
* Población Baja
* Población Media
* Población Alta

Posteriormente, se aplicó un Análisis de Varianza (ANOVA) de un factor, el cual permite comparar las medias de más de dos grupos de manera simultánea.

In [ ]:
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# 1. Segmentación de los cantones en 3 grupos equitativos
terciles = df_loja['Poblacion'].quantile([0.33, 0.66]).values

def clasificar_canton(pob):
    if pob <= terciles[0]:
        return 'Población Baja'
    elif pob <= terciles[1]:
        return 'Población Media'
    else:
        return 'Población Alta'

df_loja['Tipo_Canton'] = df_loja['Poblacion'].apply(clasificar_canton)

# 2. Ajuste del modelo lineal y cálculo del ANOVA de 1 Factor
modelo_alcantarillado = ols('Pct_Sin_Alcantarillado ~ C(Tipo_Canton)', data=df_loja).fit()
tabla_anova = sm.stats.anova_lm(modelo_alcantarillado, typ=2)

print("\n=== TABLA ANOVA DE 1 FACTOR ===")
print(tabla_anova)

# 3. Prueba de comparaciones múltiples de Tukey (Post-Hoc)
print("\n=== PRUEBA POST-HOC DE TUKEY ===")
tukey = pairwise_tukeyhsd(endog=df_loja['Pct_Sin_Alcantarillado'],
                          groups=df_loja['Tipo_Canton'],
                          alpha=0.05)
print(tukey)


=== TABLA ANOVA DE 1 FACTOR ===
                    sum_sq    df         F    PR(>F)
C(Tipo_Canton)   71.526333   2.0  0.783199  0.477356
Residual        593.617991  13.0       NaN       NaN

=== PRUEBA POST-HOC DE TUKEY ===
         Multiple Comparison of Means - Tukey HSD, FWER=0.05         
    group1          group2     meandiff p-adj   lower   upper  reject
---------------------------------------------------------------------
Población Alta  Población Baja   2.5448 0.8109 -8.2594  13.349  False
Población Alta Población Media   5.1152 0.4464  -5.689 15.9194  False
Población Baja Población Media   2.5704 0.8218 -8.7142 13.8551  False
---------------------------------------------------------------------


# 📊 **Justificación Estadística del ANOVA y Prueba Post-Hoc**

Se aplicó un ANOVA de un factor para comparar el porcentaje promedio de viviendas sin alcantarillado entre los cantones clasificados en tres grupos según su nivel de población (baja, media y alta). El análisis obtuvo un estadístico $F = 0.7832$ y un valor-p $= 0.477356$.

Como el valor-p obtenido ($0.477356$) es mayor que el nivel de significancia ($\alpha = 0.05$), **no se rechaza la hipótesis nula ($H_0$)**:

$$p\text{-value} \, (0.477356) > \alpha \, (0.05)$$

Esto indica que no existen diferencias estadísticamente significativas entre las medias de los grupos.

Además, la prueba Post-Hoc de Tukey confirmó este resultado, ya que en todas las comparaciones el resultado fue `reject = False`, evidenciando que el nivel poblacional de los cantones no influye significativamente en el porcentaje promedio de viviendas sin alcantarillado.

Este hallazgo demuestra de forma cuantitativa que las deficiencias en la infraestructura sanitaria y la falta de redes de alcantarillado constituyen una problemática transversal y homogénea en toda la provincia de Loja, afectando por igual a cantones con dinámicas demográficas altas, medias o bajas.